# Tests de mini-funcionalidades de import_trips_from_dataframe

Este notebook se usa para probar helpers y bloques internos de `import_trips_from_dataframe()` antes de hacer tests más integrados de la función completa.

Objetivo:
- verificar minifuncionalidades de forma aislada
- detectar errores de implementación temprano
- dejar una base fácil de portar después a pytest

Convenciones:
- los tests usan `assert`
- cuando una prueba necesita inspección visual, se acompaña con `display(...)`
- las pruebas de este notebook no reemplazan tests integrados ni validate_trips

## Sección 0. Preparación
- Imports
- Carga del módulo
- Helpers de apoyo para tests del notebook
- Convenciones del notebook
- (Opcional) función para resetear/revisar environment

In [1]:
import json
import math
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
from pylondrina.schema import (
    DomainSpec,
    FieldSpec,
    TripSchema,
    TripSchemaEffective,
)

from pylondrina.importing import (
    ImportOptions,
    import_trips_from_dataframe,
)

from pylondrina.datasets import TripDataset
from pylondrina.reports import ImportReport, Issue
from pylondrina.errors import ImportError as PylondrinaImportError, SchemaError, PylondrinaError

In [3]:
from pylondrina.importing import (
    _normalize_options,
    _check_schema_for_import,
    _apply_field_correspondence,
    _apply_value_correspondence,
    _get_unknown_token,
    _is_already_correct_dtype,
    _coerce_series_to_dtype,
    _coerce_columns_by_dtype,
    _normalize_source_timezone,
    _normalize_datetime_column,
    _normalize_datetime_columns,
    _normalize_hhmm_series,
    _normalize_tier2_hhmm_columns,
    _dm_to_dd,
    _dms_to_dd,
    _parse_coord_value,
    _parse_od_coordinate_columns,
    _derive_h3_indices,
    _build_keep_schema_fields,
    _select_final_columns,
    _final_required_check,
    _prune_schema_effective,
    _build_issues_summary,
    _build_import_metadata,
    _build_import_report,
    _first_required_check_and_temporal_tier,
)

In [4]:
from pylondrina.issues.core import emit_issue, emit_and_maybe_raise
from pylondrina.issues.catalog_import_trips import IMPORT_ISSUES

#### Helpers de apoyo para los tests

In [5]:
# Helper para mostrar exito simple
def show_ok(label: str):
    print(f"OK - {label}")

# Helper para serializacion JSON-safe: Util para metadata, parameters y report
def assert_json_safe(obj, label: str = "object"):
    try:
        json.dumps(obj, default=str)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e

# Helper para comparar columnas del dataframe: Muy util para seleccion final, IDs, etc.
def assert_columns_equal(df: pd.DataFrame, expected_cols, label: str = "columns"):
    actual = list(df.columns)
    expected = list(expected_cols)
    assert actual == expected, f"{label} distintas.\nActual:   {actual}\nExpected: {expected}"

# Helpers (get_issue_codes, assert_issue_present, assert_issue_absent) para verificar issues por code
def get_issue_codes(issues):
    return [i.code if hasattr(i, "code") else i.get("code") for i in issues]

def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró issue {code!r}. Codes presentes: {codes}"

def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró issue inesperado {code!r}. Codes presentes: {codes}"

# Helper para revisar dtype de columna
def assert_dtype(df: pd.DataFrame, col: str, expected_dtype: str):
    actual = str(df[col].dtype)
    assert actual == expected_dtype, f"Columna {col!r}: dtype {actual!r} != {expected_dtype!r}"

# Helper para revisar NA counts
def assert_na_count(df: pd.DataFrame, col: str, expected: int):
    actual = int(df[col].isna().sum())
    assert actual == expected, f"Columna {col!r}: n_na {actual} != {expected}"

#### Configuración visual y smoke de imports

In [6]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

print("Imports OK")
show_ok("Sección 0 cargada")

Imports OK
OK - Sección 0 cargada


## Sección 1. Utilidades puras
### Grupo 1 - utilidades base
En esta sección se prueban helpers puras o casi puras que sirven como base para bloques más grandes del import.

Objetivo:
- verificar outputs exactos,
- cubrir casos felices, bordes válidos, bordes inválidos y nulos,
- detectar errores temprano antes de probar helpers de integración.

otros
- source timezone
- unknown token
- detección de dtype correcto
- parseo de coordenadas DD/DM/DMS
- normalización HH:MM
- build_keep_schema_fields
- build_issues_summary

In [7]:
# Test de _normalize_source_timezone

tz_cases = [
    (None, (None, "none")),
    ("", (None, "empty")),
    ("   ", (None, "empty")),
    ("UTC", ("UTC", "utc")),
    ("utc", ("UTC", "utc")),
    ("Z", ("UTC", "utc")),
    ("-03:00", ("-03:00", "offset")),
    ("+00:00", ("+00:00", "offset")),
    ("America/Santiago", ("America/Santiago", "iana")),
    ("  America/Santiago  ", ("America/Santiago", "iana")),
    ("Santiago/Chile", (None, "invalid")),
    ("abc", (None, "invalid")),
]

for raw, expected in tz_cases:
    got = _normalize_source_timezone(raw)
    assert got == expected, f"_normalize_source_timezone({raw!r}) -> {got}, esperado {expected}"

show_ok("_normalize_source_timezone")

OK - _normalize_source_timezone


In [8]:
# Test de _get_unknown_token

domain_none = None
domain_unknown = DomainSpec(values=["bus", "unknown", "metro"], extendable=False, aliases=None)
domain_other = DomainSpec(values=["bus", "other", "metro"], extendable=False, aliases=None)
domain_OTHER = DomainSpec(values=["bus", "OTHER", "metro"], extendable=False, aliases=None)
domain_Unknown = DomainSpec(values=["bus", "Unknown", "metro"], extendable=False, aliases=None)
domain_no_candidate = DomainSpec(values=["bus", "metro"], extendable=False, aliases=None)
domain_empty = DomainSpec(values=[], extendable=False, aliases=None)

assert _get_unknown_token(domain_none) == "unknown"
assert _get_unknown_token(domain_unknown) == "unknown"
assert _get_unknown_token(domain_other) == "other"
assert _get_unknown_token(domain_OTHER) == "OTHER"
assert _get_unknown_token(domain_Unknown) == "Unknown"
assert _get_unknown_token(domain_no_candidate) == "unknown"
assert _get_unknown_token(domain_empty) == "unknown"

show_ok("_get_unknown_token")

OK - _get_unknown_token


In [9]:
# Test de _is_already_correct_dtype

s_string = pd.Series(["a", "b", pd.NA], dtype="string")
s_int = pd.Series([1, 2, pd.NA], dtype="Int64")
s_float = pd.Series([1.5, 2.5, np.nan], dtype="float64")
s_bool = pd.Series([True, False, pd.NA], dtype="boolean")
s_dt_naive = pd.Series(pd.to_datetime(["2026-03-01 08:00:00", None]))
s_dt_utc = pd.Series(pd.to_datetime(["2026-03-01T08:00:00Z", None], utc=True))
s_obj = pd.Series(["a", "b", None], dtype="object")

assert _is_already_correct_dtype(s_string, "string") is True
assert _is_already_correct_dtype(s_string, "categorical") is True
assert _is_already_correct_dtype(s_int, "int") is True
assert _is_already_correct_dtype(s_float, "float") is True
assert _is_already_correct_dtype(s_bool, "bool") is True
assert _is_already_correct_dtype(s_dt_naive, "datetime") is True
assert _is_already_correct_dtype(s_dt_utc, "datetime") is True

assert _is_already_correct_dtype(s_obj, "string") is False
assert _is_already_correct_dtype(s_string, "int") is False
assert _is_already_correct_dtype(s_int, "float") is False
assert _is_already_correct_dtype(s_float, "bool") is False
assert _is_already_correct_dtype(s_bool, "datetime") is False

show_ok("_is_already_correct_dtype")

OK - _is_already_correct_dtype


In [10]:
# Tests de _dm_to_dd y _dms_to_dd

# DM
assert math.isclose(_dm_to_dd(33, 30, "N"), 33.5, rel_tol=0, abs_tol=1e-9)
assert math.isclose(_dm_to_dd(70, 30, "W"), -70.5, rel_tol=0, abs_tol=1e-9)
assert math.isclose(_dm_to_dd(33, 27.6, "S"), -(33 + 27.6/60.0), rel_tol=0, abs_tol=1e-9)

# DMS
assert math.isclose(_dms_to_dd(33, 30, 0, "N"), 33.5, rel_tol=0, abs_tol=1e-9)
assert math.isclose(_dms_to_dd(70, 30, 0, "W"), -70.5, rel_tol=0, abs_tol=1e-9)
assert math.isclose(_dms_to_dd(33, 27, 36, "S"), -(33 + 27/60.0 + 36/3600.0), rel_tol=0, abs_tol=1e-9)

show_ok("_dm_to_dd / _dms_to_dd")

OK - _dm_to_dd / _dms_to_dd


In [11]:
# Tests de _parse_coord_value

val, status = _parse_coord_value(np.nan)
assert np.isnan(val)
assert status == "null"

val, status = _parse_coord_value(None)
assert np.isnan(val)
assert status == "null"

val, status = _parse_coord_value(10)
assert val == 10.0
assert status == "numeric"

val, status = _parse_coord_value(-33.45)
assert math.isclose(val, -33.45, rel_tol=0, abs_tol=1e-12)
assert status == "numeric"

val, status = _parse_coord_value("-33.446160")
assert math.isclose(val, -33.446160, rel_tol=0, abs_tol=1e-12)
assert status == "dd_direct"

val, status = _parse_coord_value("335208,7188")
assert math.isclose(val, 335208.7188, rel_tol=0, abs_tol=1e-9)
assert status == "dd_comma_decimal"

val, status = _parse_coord_value("33 27.0000 S")
assert math.isclose(val, -33.45, rel_tol=0, abs_tol=1e-9)
assert status == "dm"

dms_tests = {
    '33° 27\' 00" S': -33.45,
    "33 27 00 S" : -33.45,
    "70 30 00 W" : -70.5,
    '12 05 30.5 N' : 12.091805556,
}

for t, e in dms_tests.items():
    val, status = _parse_coord_value(t)
    assert math.isclose(val, e, rel_tol=0, abs_tol=1e-9)
    assert status == "dms"

val, status = _parse_coord_value("")
assert np.isnan(val)
assert status == "empty"

val, status = _parse_coord_value("   ")
assert np.isnan(val)
assert status == "empty"

val, status = _parse_coord_value("abc")
assert np.isnan(val)
assert status == "unparsed"

show_ok("_parse_coord_value")

OK - _parse_coord_value


In [12]:
# Tests de _normalize_hhmm_series

s_hhmm = pd.Series(
    ["08:23", "12:30", "24:00", "ab:cd", "", None, "7:05", "09:60", " 07:05 ", "00:00", "23:59"],
    dtype="object",
)

out_hhmm, stats_hhmm = _normalize_hhmm_series(s_hhmm)

assert_dtype(pd.DataFrame({"hhmm": out_hhmm}), "hhmm", "string")

# válidos preservados
assert out_hhmm.iloc[0] == "08:23"
assert out_hhmm.iloc[1] == "12:30"
assert out_hhmm.iloc[8] == "07:05"
assert out_hhmm.iloc[9] == "00:00"
assert out_hhmm.iloc[10] == "23:59"

# inválidos a NA
assert pd.isna(out_hhmm.iloc[2])   # 24:00
assert pd.isna(out_hhmm.iloc[3])   # ab:cd
assert pd.isna(out_hhmm.iloc[4])   # ""
assert pd.isna(out_hhmm.iloc[5])   # None
assert pd.isna(out_hhmm.iloc[6])   # 7:05
assert pd.isna(out_hhmm.iloc[7])   # 09:60

assert stats_hhmm["n_total"] == 11
assert stats_hhmm["n_invalid"] == 4   # 24:00, ab:cd, 7:05, 09:60
assert stats_hhmm["n_na"] == 6        # inválidos + vacíos/nulos

show_ok("_normalize_hhmm_series")

OK - _normalize_hhmm_series


In [13]:
# Tests de _build_keep_schema_fields

schema_g1 = TripSchema(
    version="0.1.0",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=False),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=False),
        "purpose": FieldSpec(name="purpose", dtype="categorical", required=False),
        "mode": FieldSpec(name="mode", dtype="categorical", required=False),
    },
    required=["movement_id", "trip_id"],
    semantic_rules=None,
)

# selected_fields=None -> todos los fields del schema
res_none = _build_keep_schema_fields(schema_g1, None)
assert res_none["schema_fields"] == {"movement_id", "trip_id", "movement_seq", "purpose", "mode"}
assert res_none["required_fields"] == {"movement_id", "trip_id"}
assert res_none["keep_schema_fields"] == {"movement_id", "trip_id", "movement_seq", "purpose", "mode"}

# selected_fields=[] -> solo required
res_empty = _build_keep_schema_fields(schema_g1, [])
assert res_empty["keep_schema_fields"] == {"movement_id", "trip_id"}

# subconjunto válido -> required ∪ selected
res_subset = _build_keep_schema_fields(schema_g1, ["purpose", "mode"])
assert res_subset["keep_schema_fields"] == {"movement_id", "trip_id", "purpose", "mode"}

# selected_fields inválido -> ValueError
try:
    _build_keep_schema_fields(schema_g1, ["purpose", "fake_field"])
    assert False, "Debió fallar por selected_fields inválido"
except ValueError:
    pass

show_ok("_build_keep_schema_fields")

OK - _build_keep_schema_fields


In [14]:
# Tests de _build_issues_summary

# caso vacío
summary_empty = _build_issues_summary([])
assert summary_empty == {"counts": {}, "by_code": {}}

issues_g1 = [
    Issue(level="warning", code="CODE_A", message="a"),
    Issue(level="warning", code="CODE_A", message="a2"),
    Issue(level="info", code="CODE_B", message="b"),
    Issue(level="error", code="CODE_C", message="c"),
]

summary_g1 = _build_issues_summary(issues_g1)

assert summary_g1["counts"] == {
    "warning": 2,
    "info": 1,
    "error": 1,
}

assert summary_g1["by_code"] == {
    "CODE_A": 2,
    "CODE_B": 1,
    "CODE_C": 1,
}

assert_json_safe(summary_g1, "issues_summary")

show_ok("_build_issues_summary")

OK - _build_issues_summary


In [15]:
show_ok("Grupo 1 completo")

OK - Grupo 1 completo


## Sección 2. Schema y options
### Grupo 2A - detección temporal

En esta subsección se prueba la detección del tier temporal y el primer chequeo estructural mínimo asociado a tiempos.

Objetivo:
- verificar que `_first_required_check_and_temporal_tier(...)` detecte correctamente `tier_1`, `tier_2` y `tier_3`,
- comprobar que emite los issues esperados,
- y confirmar que distingue entre campos faltantes derivables y no derivables.

In [17]:
# Tests de _first_required_check_and_temporal_tier

schema_temporal = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime", required=False),
        "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime", required=False),
        "origin_time_local_hhmm": FieldSpec(name="origin_time_local_hhmm", dtype="string", required=False),
        "destination_time_local_hhmm": FieldSpec(name="destination_time_local_hhmm", dtype="string", required=False),
    },
    required=["user_id", "movement_id"],
    semantic_rules=None,
)

# Caso 1: Tier 1 con UTC explícito
df_t1 = pd.DataFrame({
    "user_id": ["u1", "u2"],
    "origin_time_utc": ["2026-03-01T08:00:00Z", "2026-03-01T09:00:00Z"],
    "destination_time_utc": ["2026-03-01T08:30:00Z", "2026-03-01T09:40:00Z"],
})

issues_t1, tier_t1, fields_t1 = _first_required_check_and_temporal_tier(
    df_t1,
    schema=schema_temporal,
    single_stage=False,
    strict=False,
)

assert tier_t1 == "tier_1", f"Esperaba tier_1 y obtuvo {tier_t1!r}"
assert set(fields_t1) == {"origin_time_utc", "destination_time_utc"}
assert_issue_absent(issues_t1, "IMP.TEMPORAL.TIER_LIMITED")
# movement_id falta pero es derivable, por lo tanto no debe abortar ni emitir missing_required fatal
assert_issue_absent(issues_t1, "IMP.INPUT.MISSING_REQUIRED_FIELD")

# Caso 2: Tier 2 con HH:MM
df_t2 = pd.DataFrame({
    "user_id": ["u1", "u2"],
    "origin_time_local_hhmm": ["08:00", "09:00"],
    "destination_time_local_hhmm": ["08:30", "09:40"],
})

issues_t2, tier_t2, fields_t2 = _first_required_check_and_temporal_tier(
    df_t2,
    schema=schema_temporal,
    single_stage=False,
    strict=False,
)

assert tier_t2 == "tier_2", f"Esperaba tier_2 y obtuvo {tier_t2!r}"
assert set(fields_t2) == {"origin_time_local_hhmm", "destination_time_local_hhmm"}
assert_issue_present(issues_t2, "IMP.TEMPORAL.TIER_LIMITED")
assert_issue_absent(issues_t2, "IMP.INPUT.MISSING_REQUIRED_FIELD")

# Caso 3: Tier 3 sin información temporal OD
df_t3 = pd.DataFrame({
    "user_id": ["u1", "u2"],
    "trip_id": ["t1", "t2"],
})

issues_t3, tier_t3, fields_t3 = _first_required_check_and_temporal_tier(
    df_t3,
    schema=schema_temporal,
    single_stage=False,
    strict=False,
)

assert tier_t3 == "tier_3", f"Esperaba tier_3 y obtuvo {tier_t3!r}"
assert fields_t3 == []
assert_issue_present(issues_t3, "IMP.TEMPORAL.TIER_LIMITED")
assert_issue_absent(issues_t3, "IMP.INPUT.MISSING_REQUIRED_FIELD")

# Caso 4: DataFrame vacío -> issue EMPTY_DATAFRAME
df_empty = pd.DataFrame(columns=["user_id", "origin_time_utc", "destination_time_utc"])

issues_empty, tier_empty, fields_empty = _first_required_check_and_temporal_tier(
    df_empty,
    schema=schema_temporal,
    single_stage=False,
    strict=False,
)

assert tier_empty == "tier_1", f"Esperaba tier_1 por columnas presentes y obtuvo {tier_empty!r}"
assert set(fields_empty) == {"origin_time_utc", "destination_time_utc"}
assert_issue_present(issues_empty, "IMP.INPUT.EMPTY_DATAFRAME")

# Caso 5: Falta required no derivable -> issue MISSING_REQUIRED_FIELD
df_missing_non_derivable = pd.DataFrame({
    "origin_time_utc": ["2026-03-01T08:00:00Z"],
    "destination_time_utc": ["2026-03-01T08:30:00Z"],
})

try:
    issues_missing, tier_missing, fields_missing = _first_required_check_and_temporal_tier(
        df_missing_non_derivable,
        schema=schema_temporal,
        single_stage=False,
        strict=False,
    )
except PylondrinaImportError as error:
    display(error)
    print(error)

assert tier_missing == "tier_1"
assert_issue_present(issues_missing, "IMP.INPUT.MISSING_REQUIRED_FIELD")

show_ok("_first_required_check_and_temporal_tier")

ImportError(message="Faltan campos obligatorios para importar según el TripSchema: ['user_id'].", code='IMP.INPUT.MISSING_REQUIRED_FIELD', details={'missing_required': ['user_id'], 'required': ['user_id', 'movement_id'], 'source_columns': ['origin_time_utc', 'destination_time_utc'], 'field_correspondence_keys': [], 'field_correspondence_values_sample': []}, issue=Issue(level='error', code='IMP.INPUT.MISSING_REQUIRED_FIELD', message="Faltan campos obligatorios para importar según el TripSchema: ['user_id'].", field=None, source_field=None, row_count=None, details={'missing_required': ['user_id'], 'required': ['user_id', 'movement_id'], 'source_columns': ['origin_time_utc', 'destination_time_utc'], 'field_correspondence_keys': [], 'field_correspondence_values_sample': []}), issues=(Issue(level='error', code='IMP.INPUT.MISSING_REQUIRED_FIELD', message="Faltan campos obligatorios para importar según el TripSchema: ['user_id'].", field=None, source_field=None, row_count=None, details={'miss

ImportError: Faltan campos obligatorios para importar según el TripSchema: ['user_id'].
code: IMP.INPUT.MISSING_REQUIRED_FIELD
details:
{'missing_required': ['user_id'],
 'required': ['user_id', 'movement_id'],
 'source_columns': ['origin_time_utc', 'destination_time_utc'],
 'field_correspondence_keys': [],
 'field_correspondence_values_sample': []}
issue: Issue(level='error', code='IMP.INPUT.MISSING_REQUIRED_FIELD', message="Faltan campos obligatorios para importar según el TripSchema: ['user_id'].", field=None, source_field=None, row_count=None, details={'missing_required': ['user_id'], 'required': ['user_id', 'movement_id'], 'source_columns': ['origin_time_utc', 'destination_time_utc'], 'field_correspondence_keys': [], 'field_correspondence_values_sample': []})
issues_count: 1
issues:
(Issue(level='error',
       code='IMP.INPUT.MISSING_REQUIRED_FIELD',
       message='Faltan campos obligatorios para importar según el TripSchema: '
               "['user_id'].",
       field=None,
 

NameError: name 'tier_missing' is not defined

### Grupo 3 - schema usable para import
- fields vacío
- required vacío / inconsistente
- dtype desconocido
- categorical sin domain
- constraints inválidas

### (opcional) normalize options
- defaults
- selected_fields
- source_timezone
- h3_resolution

## Sección 3. Mappings y categóricos
### Grupo 4 - field correspondence
- mapping válido
- source faltante
- canonical inexistente
- colisiones
- conflictos con canónico ya presente

### Grupo 5 - categóricos y dominios
- normalize_cat_series
- apply_value_correspondence
- extendable=False
- extendable=True + strict_domains=False
- extendable=True + strict_domains=True
- interacción con target_schema_fields

## Sección 4. Tiempo, coerción, coordenadas y H3
### Grupo 2 - temporal (normalización)
- normalize_datetime_column
- normalize_datetime_columns
- normalize_tier2_hhmm_columns

### Grupo 6 - coerción y parseo mínimo
- coerce_series_to_dtype
- coerce_columns_by_dtype
- parse_od_coordinate_columns

### Grupo 7 - H3
- derive_h3_indices
- required_fields_unavailable
- partial_derivation

## Sección 5. IDs, selección final y required final
### Grupo 8 - IDs estructurales
- ensure_movement_id
- ensure_single_stage_ids

### Grupo 9 - selección final y poda
- select_final_columns
- final_required_check
- prune_schema_effective

## Sección 6. Metadata, report y trazabilidad
### Grupo 10 - metadata y report
- build_min_provenance
- build_import_event
- build_import_metadata
- build_import_report

## Sección 7. Smoke tests integrados
### Grupo 11 - pruebas pequeñas integradas
- happy path simple
- happy path con correspondencias
- strict_domains abort
- missing required abort
- tier_2
- single_stage=True